In [1]:
import pandas as pd

# ============ 1. BACA FILE ============
FILE = "paket_a_transaksi_pos_2024_2025.csv"
df = pd.read_csv(FILE, dtype=str)   # dibaca sebagai teks dulu agar format aneh tidak rusak
print("Jumlah baris awal:", len(df))
print("\nMissing value per kolom:")
print(df.isna().sum())

# ============ 2. HAPUS DUPLIKAT ============
df = df.drop_duplicates()
print("\nJumlah baris setelah hapus duplikat:", len(df))

# ============ 3. TANGANI MISSING VALUE ============
# kode_kasir tidak memengaruhi pendapatan, jadi diisi penanda saja
df["kode_kasir"] = df["kode_kasir"].fillna("TIDAK_DIKETAHUI")

# ============ 4. BERSIHKAN HARGA ============
# "13000" -> 13000 | "7500,00" -> 7500 | "12.000" -> 12000
def bersihkan_harga(x):
    x = str(x).strip()
    if "," in x:                        # koma = desimal, buang bagian desimalnya
        x = x.split(",")[0].replace(".", "")
    else:                               # titik = pemisah ribuan
        x = x.replace(".", "")
    return float(x)

df["harga_satuan"] = df["harga_satuan"].apply(bersihkan_harga)

# ============ 5. SERAGAMKAN TANGGAL ============
def baca_tanggal(x):
    x = x.strip()
    if x[4] == "-":                     # format 2025-06-02
        return pd.to_datetime(x, format="%Y-%m-%d")
    return pd.to_datetime(x.replace("/", "-"), format="%d-%m-%Y")   # 26/03/2025 atau 23-12-2024

df["tanggal"] = df["tanggal"].apply(baca_tanggal)
df["tahun"] = df["tanggal"].dt.year

# ============ 6. RAPIKAN KOLOM ANGKA & STATUS ============
df["jumlah"] = df["jumlah"].astype(int)
df["diskon_persen"] = df["diskon_persen"].astype(int)
df["status"] = df["status"].str.strip().str.upper()

# jumlah negatif dianggap salah ketik tanda minus -> dijadikan positif
df["jumlah"] = df["jumlah"].abs()

# jumlah > 100 dianggap outlier (jumlah normal maksimal 12) -> dibuang
outlier = df["jumlah"] > 100
print("Outlier jumlah dibuang:", outlier.sum(), "baris")
df = df[~outlier].copy()

# ============ 7. HITUNG PENDAPATAN ============
# pendapatan = jumlah x harga x (1 - diskon/100)
df["pendapatan"] = df["jumlah"] * df["harga_satuan"] * (1 - df["diskon_persen"] / 100)

# hanya transaksi SELESAI yang dihitung (BATAL dan RETUR tidak menjadi pendapatan)
selesai = df[df["status"] == "SELESAI"]

hasil = selesai.groupby("tahun")["pendapatan"].sum().reset_index()
hasil.columns = ["Tahun", "Total Pendapatan (Rp)"]
hasil["Total Pendapatan (Rp)"] = hasil["Total Pendapatan (Rp)"].astype(int)

print("\n=== PENDAPATAN PER TAHUN (status SELESAI, setelah diskon) ===")
print(hasil.to_string(index=False))

# ============ 8. SIMPAN HASIL ============
hasil.to_csv("hasil_pendapatan_per_tahun.csv", index=False)
print("\nHasil disimpan ke hasil_pendapatan_per_tahun.csv")

Jumlah baris awal: 5030

Missing value per kolom:
id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64

Jumlah baris setelah hapus duplikat: 5000
Outlier jumlah dibuang: 12 baris

=== PENDAPATAN PER TAHUN (status SELESAI, setelah diskon) ===
 Tahun  Total Pendapatan (Rp)
  2024              128048525
  2025              136467375

Hasil disimpan ke hasil_pendapatan_per_tahun.csv
